### Stratified Cross Validation: Handling Class Imbalance

#### 1. Problem Kya Hai? (Class Imbalance & Normal CV)
Maan lo tum ek Fraud Detection model bana rahe ho jisme 100 transactions hain: **90 Normal aur 10 Fraud**. 
Agar tum isme normal `KFold` (Cross Validation) lagate ho aur data ko 5 hisso me todte ho (20-20 ke chunks me), toh ho sakta hai ek fold me galti se **saare 20 Normal transactions** aa jayein aur ek bhi Fraud na aaye! 
Agar us fold par model test hoga, toh wo Fraud ke baare me kuch seekh hi nahi payega aur training barbad ho jayegi.

#### 2. Solution: Stratification Kya Karta Hai?
**Stratified** ka seedha sa matlab hai **"Ratio Maintain Rakhna"**. 
Ye CV karte waqt is baat ka dhyan rakhta hai ki pure dataset me classes ka jo percentage/ratio hai, wahi same ratio tumhare har ek Fold (Train/Test split) me bhi hona chahiye. 
*Example:* Agar pure data me Fraud 10% hai, toh Stratified CV ensure karega ki har ek fold me exactly 10% data Fraud ka hi ho.

#### 3. Scikit-learn ke 3 Stratified APIs

**A. `StratifiedKFold`**
*   *Kaam:* Ye data ko $K$ equal hisso me todta hai, par har hisse me class ratio same rakhta hai.
*   *Kab use karein:* Ye Classification problems ke liye absolute standard (default) tareeka hai. Ise aankh band karke use kar sakte ho.

**B. `RepeatedStratifiedKFold`**
*   *Kaam:* Ye `StratifiedKFold` hi hai, bas usi process ko $N$ baar alag-alag random splits ke sath repeat karta hai.
*   *Kab use karein:* Jab tumhara dataset bahut hi chhota ho aur tumhe apne model ki accuracy par 100% confidence chahiye ho. (Ye time thoda zyada leta hai).

**C. `StratifiedShuffleSplit`**
*   *Kaam:* Ye pure data ko shuffle karta hai, aur randomly ek Train/Test set nikal leta hai (ratio maintain karte hue).
*   *Important Note:* Normal K-Fold me ek data point sirf ek hi baar Test set me jata hai. Lekin `StratifiedShuffleSplit` me data randomly pick hota hai, isliye **ek hi data point multiple times Test set me aa sakta hai**. (Yahi tumhari slide ke note me likha hai ki folds completely alag/mutually exclusive nahi hote).

---

In [10]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, StratifiedKFold

x, y = load_iris(as_frame=True, return_X_y=True)

x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.2,
    random_state=42
)

s_kfold = StratifiedKFold(n_splits=5)

for train_index, val_index in s_kfold.split(x_train, y_train):

    x_fold_train = x_train.iloc[train_index]
    x_fold_val = x_train.iloc[val_index]

    y_fold_train = y_train.iloc[train_index]
    y_fold_val = y_train.iloc[val_index]

    print("Train size:", len(train_index))
    print("Validation size:", len(val_index))
    print("-" * 30)

Train size: 96
Validation size: 24
------------------------------
Train size: 96
Validation size: 24
------------------------------
Train size: 96
Validation size: 24
------------------------------
Train size: 96
Validation size: 24
------------------------------
Train size: 96
Validation size: 24
------------------------------


In [1]:

import numpy as np
from sklearn.model_selection import StratifiedKFold

# Dummy Imbalanced Data: 10 samples (8 Normal (0), 2 Fraud (1))
X = np.zeros(10) # X doesn't matter here, just need shapes
y = np.array([0, 0, 0, 0, 0, 0, 0, 0, 1, 1])

# Initialize StratifiedKFold (2 splits)
# Ratio in y is 8:2 (4:1). So every split should have 4 Normal and 1 Fraud!
skf = StratifiedKFold(n_splits=2)

fold_no = 1
for train_index, test_index in skf.split(X, y):
    print(f"--- Fold {fold_no} ---")
    print("Test labels (y):", y[test_index])
    fold_no += 1

# Output me tum dekhoge ki har test set me exactly ek '1' (Fraud) aayega.
# Ye normal KFold guarantee nahi kar sakta tha.

--- Fold 1 ---
Test labels (y): [0 0 0 0 1]
--- Fold 2 ---
Test labels (y): [0 0 0 0 1]


### LogisticRegressionCV: Auto-Tuning Your Model

#### 1. Ye Kaam Kaise Karta Hai?
Is estimator ke andar ek in-built machinery hoti hai jo cross-validation (folds) ka use karke tumhare liye sabse best Hyperparameters (jaise $C$, jo regularization control karta hai) automatically dhoondh leti hai. 

#### 2. Iske 4 Sabse Important Parameters:
*   **`Cs` (Regularization Strengths):** Yahan tum values ki ek list dete ho (ya ek number like 10). Ye list un saare $C$ values ki hoti hai jinko model "test" karke dekhna chahta hai. *(Yaad hai na? Chhota C = strong penalty, Bada C = weak penalty)*.
*   **`cv` (Cross-Validation Iterator):** Ye batata hai ki data ko kitne folds (hisson) me todna hai. Yahan tum seedha ek number (like `5`) de sakte ho, ya pichli slide wala `StratifiedKFold` ka object pass kar sakte ho.
*   **`scoring` (Scoring Function):** Model test karte waqt usko pata kaise chalega ki "Best" kon hai? Yaha hum metrics specify karte hain jaise `'accuracy'`, `'f1'`, `'roc_auc'`.

#### 3. The Magic of `refit` (Interview me poonchte hain!)
Sabse zyada confusion `refit` parameter me hoti hai. Jab model CV ke dauran har fold pe alag-alag $C$ try kar leta hai, uske baad kya hota hai? Yaha 2 options hain:

*   **`refit = True` (Default aur sabse best tarika):**
    *   *Logic:* Ye dekhta hai ki saare folds ka average score nikalne ke baad kaunsa **$C$** sabse best nikla. 
    *   *Action:* Fir ye us best $C$ ko uthata hai, aur baaki saare kachre ko bhool kar **poore ke poore dataset (100% data) par ek ekdam fresh naya model train (refit) karta hai**. Ye sabse accurate aur safe approach hai.

*   **`refit = False` (Fast but Shortcut tarika):**
    *   *Logic:* Ye dobara se poore data par fresh model train nahi karta.
    *   *Action:* Har fold me jo best models bane the, unke **Weights (coefs)**, **Intercepts**, aur **Best C** ko ye aapas me simply **Average (mean)** kar deta hai, aur usi ko final model bana deta hai. 
    *   *Kab use karein:* Jab data itna bada ho ki ek baar dobara train karne me ghanto lag jayein, tab ye time bachane ka jugaad hai.
---
### Classification Metrics: Minimal Cheatsheet

Scikit-learn ka `sklearn.metrics` module true labels ($y\_true$) aur predicted labels ($y\_pred$) ko compare karke model ki performance batata hai. 

#### 1. Core Metrics (Kab konsa dekhein?)

*   **`accuracy_score`**: Total kitne % predictions sahi hain.
    *   *Rule:* Sirf tab use karo jab data perfectly balanced ho. (Imbalanced me ye jhooth bolta hai).
*   **`balanced_accuracy_score`**: Har class ki accuracy nikal kar unka average karta hai.
    *   *Rule:* Imbalanced datasets (jaise 90% normal, 10% fraud) me hamesha ye use karo.
*   **`precision_score`**: Model ne jo "Positive" predict kiya, usme se sach me kitne positive the?
    *   *Focus:* False Positives (Galat Alarms) ko kam karna hai (e.g., Spam filter).
*   **`recall_score`**: Asal me jo "Positive" the, usme se model kitne dhoondh paya?
    *   *Focus:* False Negatives (Miss na ho) ko kam karna hai (e.g., Cancer detection).
*   **`f1_score`**: Precision aur Recall ka harmonic mean. Dono me balance banata hai.
*   **`roc_auc_score`**: Model True Positives aur False Positives ke beech kitna achha discriminate (alag) kar pata hai. (Probability outputs pe kaam karta hai).
*   **`top_k_accuracy_score`**: Model ki top 'k' predictions me se kya actual class maujood hai? (Multiclass me use hota hai, e.g., Top 3 recommendations).

#### 2. Universal Syntax Pattern
Lagbhag saare metrics ek hi format follow karte hain: 
`metric_name(actual_labels, predicted_labels)`

```python
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Assume model is already trained
y_pred = model.predict(x_test)

# Calculate metrics
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average='macro') # average='macro' zaroori hai multiclass ke liye
rec = recall_score(y_test, y_pred, average='macro')
f1 = f1_score(y_test, y_pred, average='macro')

print(f"Accuracy: {acc:.2f} | Precision: {prec:.2f} | Recall: {rec:.2f} | F1: {f1:.2f}")
```
---

In [11]:

from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import StratifiedKFold

# Stratified Fold banaya taaki class imbalance handle ho
cv_strategy = StratifiedKFold(n_splits=5)

# Initialize LogisticRegressionCV
# Cs=10 means ye apne aap 10 alag-alag C values (from 1e-4 to 1e4) try karega
log_cv_model = LogisticRegressionCV(
    Cs=10, 
    cv=cv_strategy, 
    scoring='accuracy', 
    refit=True, # Will retrain a fresh model on best C
    max_iter=1000,
    random_state=42
)

# Fit karte hi ye CV aur Tuning dono ek sath shuru kar dega
log_cv_model.fit(x_train, y_train)

# Training ke baad tum directly best parameters dekh sakte ho!
print("Sabse best C value jo select hui:", log_cv_model.C_)

c:\Users\sumit\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
c:\Users\sumit\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegressionCV for more details.
  warnings.w

Sabse best C value jo select hui: [0.35938137 0.35938137 0.35938137]
